**Getting the dataset using an URL from the huggingface**

In [3]:
import pandas as pd
url = "https://huggingface.co/datasets/JEFFREY-VERDIERE/Creditcard/resolve/feac26da102fcef433202e29a528e3bfb6a62a04/creditcard.csv"
df = pd.read_csv(url)
print(df.shape)

(284807, 31)


In [7]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Checking the number of frauds
num_frauds = int(df["Class"].sum())
print(df.shape, "| fraud cases:", num_frauds)

scaler = StandardScaler()
df["Amount"] = scaler.fit_transform(df[["Amount"]])
df["Time"] = scaler.fit_transform(df[["Time"]])

# features and labels
X = df.drop("Class", axis=1).values
y = df["Class"].values

X_normal = X[y == 0]
X_train, X_test_normal = train_test_split(X_normal, test_size=0.2, random_state=42)
X_fraud = X[y == 1]

X_test = np.vstack([X_test_normal, X_fraud])
y_test = np.hstack([np.zeros(len(X_test_normal)), np.ones(len(X_fraud))])
print("train shape:", X_train.shape, "| test shape:", X_test.shape, "| frauds in test:", int(y_test.sum()))

(284807, 31) | fraud cases: 492
train shape: (227452, 30) | test shape: (57355, 30) | frauds in test: 492


# Model A: Autoencoder (AE)

In [8]:
import tensorflow as tf
from tensorflow.keras import layers, Model

dim = X_train.shape[1]

ae = tf.keras.Sequential([
    layers.Input(shape=(dim,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(dim, activation="linear"),
])
ae.compile(optimizer="adam", loss="mse")

# training AE for 10 epochs
ae.fit(X_train, X_train, epochs=10, batch_size=256, validation_split=0.1, verbose=1)

Epoch 1/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.6582 - val_loss: 0.4716
Epoch 2/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.4135 - val_loss: 0.3929
Epoch 3/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.3610 - val_loss: 0.3492
Epoch 4/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.3263 - val_loss: 0.3236
Epoch 5/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.3111 - val_loss: 0.3128
Epoch 6/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.3033 - val_loss: 0.3064
Epoch 7/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.2985 - val_loss: 0.3033
Epoch 8/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.2954 - val_loss: 0.2997
Epoch 9/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.2920 - val_loss: 0.2966
Epoch 10/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.2882 - val_loss: 0.2908


In [9]:
from sklearn.metrics import roc_auc_score

recon = ae.predict(X_test)
ae_error = np.mean((X_test - recon) ** 2, axis=1)   # reconstruction error per transaction
ae_auc = roc_auc_score(y_test, ae_error)            # higher error should mean fraud
print("Autoencoder AUC:", round(ae_auc, 3))

1793/1793 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
Autoencoder AUC: 0.968
